# 04 · Production Python: what belongs outside a notebook?
**⏱ 9 minutes · From Maven/Gradle and JUnit to uv, packages and pytest**

So far we've written Python in memory. Real AI services need more: dependencies, reusable modules, tests, logging, configuration and predictable environments.

If you already build Spring Boot services, you know why. The difference isn't **why** we need these tools. It's **how** Python projects put them together. This is the lesson that prevents your notebook experiments from becoming an unmaintainable pile of cells.

## 1. Compare a Java project with this Python repository

A typical Java project has `pom.xml` or `build.gradle`, source folders and a test tree. Python projects commonly use a `pyproject.toml` file, a `src/` package, and a `tests/` directory.

```text
Java                         Python
pom.xml / build.gradle       pyproject.toml
src/main/java/...            src/mini_agent/...
src/test/java/...            tests/...
Maven/Gradle                 uv (environment + dependency management)
JUnit                        pytest
```

This repository **actually contains** a small package called `mini_agent`. The course installs it from the `src/` directory in editable mode with `uv sync`.

**Check:** which Python files here correspond to a Java DTO, a service layer, and tests?

In [ ]:
from pathlib import Path
import mini_agent
from mini_agent import Ticket, classify_ticket

print("Imported package from:", Path(mini_agent.__file__).name)
item = Ticket(id=10, subject="Login problem", body="Cannot log in")
print("Classification:", classify_ticket(item))
assert classify_ticket(item) == "access"

## 2. Understand uv rather than copying commands

In Java, dependency versions are resolved by your build tool. Python also needs a **specific interpreter and an isolated dependency environment**.

- `uv python install 3.12`: install a known compatible Python version.
- `uv sync --group dev`: create/update `.venv`, install your package and developer tools.
- `uv run python ...`: run inside the project's environment without manually activating it.
- `uv run pytest -q`: run tests using that same environment.

In your editor, select **this project's `.venv` interpreter**. Using system Python is a common cause of `ModuleNotFoundError`.

**Try these in a TERMINAL, not in a notebook cell:**

```bash
uv python install 3.12
uv sync --group dev
uv run python -c "import mini_agent, pydantic; print('ready')"
uv run pytest -q
```

More detail: [uv setup and debugging guide](../guides/05_uv_and_debugging.md).

## 3. Imports and module boundaries

Java packages group classes; Python **modules** are `.py` files and **packages** are importable collections of modules.

For example, the project has these files:

```text
src/mini_agent/
    __init__.py       # public import surface
    models.py         # Pydantic contracts
    tools.py          # classification + fake I/O
    workflow.py       # orchestration + approval simulation
```

When you type `from mini_agent import Ticket`, Python imports a name exposed by `mini_agent/__init__.py`. Avoid writing all your functionality inside notebooks; export reusable code from modules instead.

**Prediction:** what is the difference between the `Ticket` *class* and an actual `Ticket(...)` *instance*?

In [ ]:
from mini_agent import Ticket

print("Class:", Ticket.__name__)
example = Ticket(id=20, subject="Billing help", body="Invoice is missing")
print("Instance:", example.model_dump())
assert example.id == 20

## 4. Files and context managers

Java's `try-with-resources` closes a file at the end of a block. Python has **context managers** for the same kind of lifetime control. The keyword is `with`; `pathlib.Path` gives you a convenient object for file paths.

We use a **temporary directory** so running this lesson doesn't modify the repository. Try changing the text and watch the cleanup message.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    file_path = Path(folder) / "example.txt"
    file_path.write_text("hello from Python", encoding="utf-8")
    assert file_path.read_text(encoding="utf-8") == "hello from Python"
    print("File exists inside block:", file_path.exists())

print("Temporary directory cleaned up automatically")

## 5. Testing: a quick JUnit ↔ pytest comparison

**Java / JUnit:**

```java
@Test void rejectsNegativeId() {
    assertThrows(IllegalArgumentException.class,
        () -> new Ticket(-1, "Example"));
}
```

**Python / pytest:**

```python
import pytest
from pydantic import ValidationError

with pytest.raises(ValidationError):
    Ticket(id=-1, subject="Test", body="Test body")
```

Tests catch **behavioral regressions**. Type hints can help a linter, Pydantic checks data when it's constructed, and pytest checks the **actual behavior**. These are complementary, not interchangeable.

Here is the exact kind of failing-input check the project tests use.

In [ ]:
from pydantic import ValidationError

try:
    Ticket(id=-1, subject="Problem", body="Long enough body")
except ValidationError:
    print("PASS: invalid ID rejected at the application boundary")
else:
    raise AssertionError("Pydantic was expected to reject negative IDs")

### Your production checklist

**Configuration:** read API keys from environment variables or a secret manager, never hardcode them in notebooks or commit `.env` files.

**Reproducibility:** `pyproject.toml` defines dependency ranges; a committed `uv.lock` fixes resolved versions for repeated installs. This repository's CI verifies its declared setup; see the [guide](../guides/05_uv_and_debugging.md) for lockfile practices.

**Diagnostics:** in an app use structured logging instead of relying only on prints. Be cautious about logging sensitive prompts or model responses.

**Checkpoint:** explain why `uv run pytest -q` and `uv run python scripts/validate_notebooks.py` test different things.

**Next:** [05 · Assemble the mini AI workflow](05_mini_ai_agent.ipynb). Additional reference: [Docker, serialization and other useful foundations](../guides/07_extra_python_foundations.md).